# 第13回：教師なし学習と次元圧縮・特徴量選択

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/DS_Seminar1_Public/blob/main/notebooks/session13_unsupervised_dimensionality_feature_selection.ipynb)

**DSゼミナールⅠ（2026年度）**  
熊本大学 データサイエンス学科

第12回までは、正解ラベルがある分類・回帰モデルを作り、混同行列、ROC曲線、PR曲線、残差などで評価しました。今回は、正解ラベルを最初から使わずにデータの構造を調べる**教師なし学習**、多くの特徴量を少数の軸に要約する**次元圧縮**、モデルに使う特徴量を整理する**特徴量選択**を扱います。

同じ `winequality.csv` を使いますが、今回の主役は「予測精度」だけではありません。データの中にどのようなまとまりがあるか、どの特徴量が似た情報を持っているか、少ない特徴量でも分析できるかを確認します。

---

## 📋 本日の目標

1. 教師あり学習と教師なし学習の違いを説明できる
2. K-meansクラスタリングの基本的な考え方を説明できる
3. PCAによる次元圧縮と寄与率の意味を説明できる
4. t-SNEによる非線形な可視化の特徴と注意点を説明できる
5. 特徴量選択の目的と代表的な方法を説明できる
6. `scikit-learn` を使ってクラスタリング、PCA、t-SNE、特徴量選択を実装できる

---

## 0. 準備：ライブラリの読み込み


In [ ]:
import sys
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

try:
    import japanize_matplotlib
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "japanize-matplotlib", "-q"])
    import japanize_matplotlib

sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "IPAexGothic"
plt.rcParams["axes.unicode_minus"] = False


---

## 1. 教師なし学習とは

教師あり学習では、入力 `X` と正解 `y` の対応を学習します。

| 種類 | 正解ラベル | 代表例 | 目的 |
| ---- | ---- | ---- | ---- |
| 教師あり学習 | ある | 分類、回帰 | 正解を予測する |
| 教師なし学習 | ない | クラスタリング、次元圧縮 | データの構造を見つける |

教師なし学習では、モデルに「赤ワイン」「白ワイン」「高品質」「低品質」といった答えを最初から教えません。特徴量だけを見て、似ているサンプルをまとめたり、複雑なデータを見やすい形に変換したりします。

### 1-1. 何に使うのか

- データの全体像を把握する
- 似ているサンプル同士をグループ化する
- 外れ値や特殊なパターンを見つける
- 多数の特徴量を2次元や3次元に要約して可視化する
- 予測モデルを作る前に、特徴量の重複や情報量を確認する

### 1-2. 注意点

教師なし学習の結果は、必ずしも医学的・社会的・実務的な意味を持つとは限りません。たとえば、K-meansで3つのクラスタが出たとしても、それが本当に3種類の疾患サブタイプや3種類の顧客タイプを意味するとは限りません。

結果を解釈するときは、次の点を確認します。

- どの特徴量がクラスタの違いに関係しているか
- 標準化の有無で結果が変わらないか
- クラスタ数を変えたときに解釈が安定するか
- 後から既知ラベルを重ねたときに意味のある分かれ方になっているか

---

## 2. データの読み込み

第12回と同じ `data/winequality.csv` を使います。


In [ ]:
df = pd.read_csv("data/winequality.csv")

feature_cols = [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
]

X = df[feature_cols]
y_type = (df["type"] == "red").astype(int)
y_quality = df["quality"]

print(df.shape)
df.head()


今回、クラスタリングやPCAでは原則として `X` だけを使います。`type` や `quality` は、結果を後から解釈するための補助情報として使います。

### 2-1. 特徴量のスケールを確認する


In [ ]:
X.describe().T


特徴量ごとに単位や範囲が大きく違います。K-meansやPCAは距離や分散に基づくため、標準化が重要です。


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)
X_scaled_df.describe().T[["mean", "std", "min", "max"]]


---

## 3. クラスタリング：K-means

クラスタリングは、似ているデータを同じグループにまとめる方法です。正解ラベルを使わずに、「このデータ同士は近そう」「このデータは別のまとまりに入りそう」といった構造を探します。

K-meansは、クラスタリングの代表的な方法です。最初に「いくつのグループに分けたいか」を決めてから、データを近いもの同士に分けていきます。

### 3-1. K-meansの考え方

K-meansでは、あらかじめクラスタ数 `K` を決めます。たとえば `K=2` なら2グループ、`K=3` なら3グループに分けます。

考え方は次のようにシンプルです。

1. 適当な場所にグループの中心を置く
2. 各データを、一番近い中心のグループに入れる
3. グループに入ったデータの平均を、新しい中心にする
4. グループ分けがあまり変わらなくなるまで繰り返す

つまりK-meansは、「近い中心に集める」と「中心を置き直す」を繰り返して、データを自然なまとまりに分ける方法です。

たとえば `K=3` の場合は、データを3つのグループに分けます。2つの特徴量で表すと、次のようなイメージです。

```text
特徴量2
  ↑
  |        クラスタ1
  |       ○ ○ ○
  |      ○  ★  ○             クラスタ2
  |       ○ ○ ○             △ △ △
  |                         △  ★  △
  |                          △ △ △
  |
  |              クラスタ3
  |             □ □ □
  |            □  ★  □
  |             □ □ □
  +--------------------------------→ 特徴量1

★：クラスタの中心
○ △ □：各データ
```

K-meansでは、各データを一番近い `★` に集めます。その後、各グループの平均位置に `★` を置き直します。この作業を繰り返すことで、最終的に3つのクラスタができます。

### 3-2. 今回使うK-means関連のライブラリ

この章では、次のライブラリを使います。


In [ ]:
from sklearn.cluster import KMeans


| ライブラリ | 役割 | この授業での使い方 |
| ---- | ---- | ---- |
| `KMeans` | データを指定した数のクラスタに分ける | ワインの成分データを、ラベルなしで2つのグループに分ける |

`KMeans(n_clusters=2)` と書くと、データを2つのクラスタに分けます。`fit_predict(X_scaled)` を実行すると、学習とクラスタ番号の取得を同時に行えます。

- `n_clusters`：いくつのクラスタに分けるか
- `random_state`：結果を再現しやすくするための乱数固定
- `n_init`：初期値を変えて何回試すか

### 3-3. K=2でクラスタリングする


In [ ]:
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster = kmeans.fit_predict(X_scaled)

df_result = df.copy()
df_result["cluster"] = cluster

pd.crosstab(df_result["cluster"], df_result["type"], normalize="index")


この表では、各クラスタに赤ワインと白ワインがどの程度含まれているかを確認します。ただし、K-meansは `type` を知らずに学習しているため、クラスタ番号そのものには「赤」「白」という意味はありません。

### 3-4. クラスタごとの特徴量平均を見る


In [ ]:
cluster_profile = df_result.groupby("cluster")[feature_cols].mean().T
cluster_profile


In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(cluster_profile, cmap="coolwarm", center=X[feature_cols].mean().mean())
plt.title("クラスタごとの特徴量平均")
plt.ylabel("特徴量")
plt.xlabel("クラスタ")
plt.show()


平均値を見ることで、クラスタ0とクラスタ1がどのような特徴量で違っているかを考えます。

---

## 4. クラスタ数の決め方：エルボー法

K-meansでは、クラスタ数 `K` を自分で決める必要があります。エルボー法では、`K` を増やしたときにクラスタ内のばらつきがどの程度減るかを見ます。


In [ ]:
inertias = []
K_range = range(1, 11)

for k in K_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(K_range, inertias, marker="o")
plt.xlabel("クラスタ数 K")
plt.ylabel("Inertia")
plt.title("エルボー法")
plt.xticks(list(K_range))
plt.grid(True)
plt.show()


`inertia_` は、各データ点から所属クラスタの重心までの距離の二乗和です。`K` を増やすほど小さくなりますが、あるところから改善が緩やかになります。その折れ曲がりに見える場所を候補にします。

> エルボー法は目安です。明確な折れ曲がりが出ない場合もあります。実務では、解釈しやすさ、再現性、目的に合うかも含めて判断します。

---

## 5. 次元圧縮：PCA

PCA（主成分分析）は、多数の特徴量を少数の新しい軸に変換する方法です。元の特徴量をそのまま選ぶのではなく、特徴量の組み合わせから新しい軸を作ります。

イメージとしては、元の特徴量に重みをかけて足し合わせ、新しい合成特徴量を作る処理です。

`PC1 = a1 × fixed_acidity + a2 × volatile_acidity + ... + a11 × alcohol`

`PC2 = b1 × fixed_acidity + b2 × volatile_acidity + ... + b11 × alcohol`

ここで、`PC1` は第1主成分、`PC2` は第2主成分です。`a1, a2, ...` や `b1, b2, ...` はPCAによって決まる重みです。つまりPCAでは、11個のワイン成分を直接1つずつ見るのではなく、「酸味・糖分・アルコールなどを合成した新しい軸」としてデータを見直します。

より一般的には、標準化した特徴量を `x1, x2, ..., xp` とすると、第1主成分は次のように表せます。

`PC1 = w1 x1 + w2 x2 + ... + wp xp`

この重み `w1, w2, ..., wp` を見ることで、どの特徴量がその主成分に強く関係しているかを解釈できます。

### 5-1. PCAのイメージ

たとえば11個のワイン成分があると、散布図で全体を直接見るのは難しいです。PCAを使うと、情報のばらつきが大きい方向から順に、第1主成分、第2主成分として要約できます。

```mermaid
graph LR
    A[11個の特徴量] --> B[PCA]
    B --> C[第1主成分]
    B --> D[第2主成分]
    C --> E[2次元散布図]
    D --> E
```

### 5-2. 今回使うPCA関連のライブラリ

この章では、次のライブラリを使います。


In [ ]:
from sklearn.decomposition import PCA


| ライブラリ | 役割 | この授業での使い方 |
| ---- | ---- | ---- |
| `PCA` | 多数の特徴量を少数の主成分に変換する | 11個のワイン特徴量を `PC1` と `PC2` の2次元に圧縮する |

`PCA(n_components=2)` と書くと、元の特徴量を2つの主成分に変換します。`fit_transform(X_scaled)` を実行すると、PCAの軸を学習し、その軸に沿った新しい座標を取得できます。

- `n_components`：いくつの主成分に圧縮するか
- `explained_variance_ratio_`：各主成分が元データのばらつきをどれだけ説明しているか
- `components_`：各主成分を作るときの特徴量の重み

### 5-3. PCAを実行する


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["type"] = df["type"]
pca_df["quality"] = df["quality"]
pca_df["cluster"] = cluster

print("寄与率:", pca.explained_variance_ratio_)
print("累積寄与率:", pca.explained_variance_ratio_.sum())


寄与率は、その主成分が元データのばらつきをどの程度説明しているかを示します。第1主成分と第2主成分の累積寄与率が高いほど、2次元の図に多くの情報が残っていると考えられます。

### 5-4. PCAの散布図


In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="type",
    alpha=0.7
)
plt.title("PCAによる2次元可視化（色：ワイン種別）")
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="cluster",
    palette="Set2",
    alpha=0.7
)
plt.title("PCAによる2次元可視化（色：K-meansクラスタ）")
plt.show()


1つ目の図では、後から `type` を重ねています。2つ目の図では、K-meansのクラスタを重ねています。教師なし学習の結果が既知ラベルとどの程度対応しているかを確認できます。

### 5-5. 主成分負荷量を見る

PCAの軸は、元の特徴量の組み合わせです。どの特徴量が各主成分に強く関係しているかは、負荷量で確認します。


In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_cols,
    columns=["PC1", "PC2"]
)

loadings.sort_values("PC1", key=abs, ascending=False)


In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(loadings, annot=True, cmap="coolwarm", center=0)
plt.title("PCA負荷量")
plt.show()


負荷量の絶対値が大きい特徴量ほど、その主成分に強く関係しています。符号が同じ特徴量は同じ方向に、符号が逆の特徴量は反対方向に効いていると解釈します。

---

## 6. 発展：t-SNEによる非線形な可視化

PCAは、元の特徴量を線形に組み合わせて新しい軸を作る方法でした。そのため、`PC1` や `PC2` の負荷量を見ることで、軸の意味をある程度解釈できます。

一方、t-SNEは、データ同士の「近さ」をできるだけ保ちながら2次元に配置する方法です。PCAよりも複雑なまとまりが見えやすいことがあります。

| 方法 | 得意なこと | 注意点 |
| ---- | ---- | ---- |
| PCA | 軸の意味を解釈しやすい | 線形な関係を中心に見る |
| t-SNE | 近いデータ同士のまとまりを可視化しやすい | 軸の意味やクラスタ間の距離は解釈しにくい |

### 6-1. t-SNEの考え方

t-SNEでは、元の高次元空間で近いデータ同士が、2次元の図でも近くに来るように配置します。

```mermaid
graph LR
    A[11個の特徴量] --> B[t-SNE]
    B --> C[2次元の配置]
    C --> D[近いデータ同士のまとまりを見る]
```

ただし、t-SNEの横軸・縦軸そのものには、PCAの `PC1` や `PC2` のような明確な意味はありません。図の形は `perplexity` や `random_state` などの設定でも変わります。

### 6-2. 今回使うt-SNE関連のライブラリ

この章では、次のライブラリを使います。


In [ ]:
from sklearn.manifold import TSNE


| ライブラリ | 役割 | この授業での使い方 |
| ---- | ---- | ---- |
| `TSNE` | 高次元データを2次元や3次元に配置する | 11個のワイン特徴量を2次元に配置し、赤/白ワインのまとまりを見る |

`TSNE(n_components=2)` と書くと、データを2次元に配置します。PCAと違い、`TSNE` は主に可視化のために使います。

- `n_components`：何次元に配置するか
- `perplexity`：近くのデータをどのくらい広く見るかを調整する値
- `learning_rate`：配置を更新するときの学習率
- `init`：初期配置の方法。ここではPCAの結果を初期値にする
- `random_state`：結果を再現しやすくするための乱数固定

### 6-3. t-SNEを実行する

t-SNEは計算に時間がかかるため、ここでは最大2000件をサンプリングして実行します。


In [ ]:
sample_df = df.sample(n=min(2000, len(df)), random_state=42)
sample_X = sample_df[feature_cols]
sample_X_scaled = scaler.transform(sample_X)

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=42
)
X_tsne = tsne.fit_transform(sample_X_scaled)

tsne_df = pd.DataFrame(X_tsne, columns=["TSNE1", "TSNE2"])
tsne_df["type"] = sample_df["type"].values
tsne_df["quality"] = sample_df["quality"].values


### 6-4. t-SNEの散布図


In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=tsne_df,
    x="TSNE1",
    y="TSNE2",
    hue="type",
    alpha=0.7
)
plt.title("t-SNEによる2次元可視化（色：ワイン種別）")
plt.show()


t-SNEの図では、近くにある点同士は似ている可能性があります。一方で、離れたクラスタ同士の距離や、左右・上下の方向を強く解釈しすぎないように注意します。

### 6-5. PCAとt-SNEの使い分け

- PCAは、軸の意味や特徴量の寄与を説明したいときに使いやすい
- t-SNEは、データのまとまりを視覚的に確認したいときに使いやすい
- 論文やレポートでは、t-SNEの図だけで結論を出さず、元の特徴量や別の解析結果と合わせて解釈する

---

## 7. 特徴量選択

特徴量選択は、モデルに使う特徴量を絞る処理です。特徴量が多すぎると、次のような問題が起こることがあります。

- モデルが複雑になり、過学習しやすい
- 計算時間が増える
- 結果の説明が難しくなる
- 似た情報を持つ特徴量が重複する

特徴量選択の目的は、単に特徴量数を減らすことではありません。予測性能、解釈しやすさ、再現性のバランスを取ることです。

### 7-1. 代表的な方法

| 方法 | 例 | 特徴 |
| ---- | ---- | ---- |
| フィルタ法 | 相関、ANOVA、相互情報量 | モデルとは独立に特徴量を評価する |
| ラッパー法 | RFE | モデルを何度も学習して特徴量を選ぶ |
| 組み込み法 | Lasso、Random Forest重要度 | モデル学習の中で重要度を得る |

今回は、フィルタ法とRandom Forestの重要度を確認します。

### 7-2. 今回使う特徴量選択関連のライブラリ

この章では、主に次のライブラリを使います。


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier


| ライブラリ | 役割 | この授業での使い方 |
| ---- | ---- | ---- |
| `SelectKBest` | スコアが高い特徴量を上位 `k` 個選ぶ | 11個のワイン特徴量から、赤/白分類に関係が強そうな5個を選ぶ |
| `f_classif` | 分類問題用に、各特徴量とクラスの関係をスコア化する | `SelectKBest(score_func=f_classif, k=5)` の形で使う |
| `RandomForestClassifier` | 多数の決定木を使う分類モデル | 学習後の `feature_importances_` で特徴量重要度を見る |

`SelectKBest` と `f_classif` は、モデルを本格的に作る前に、各特徴量を個別に評価する方法です。たとえば、`alcohol` や `density` が赤/白の違いと強く関係していれば、高いスコアになります。

`SelectKBest` では、`score_func` に「特徴量をどう採点するか」を指定できます。今回の `f_classif` 以外にも、目的に応じて次のような関数を選べます。

| `score_func` | 主な用途 | 見ているもの |
| ---- | ---- | ---- |
| `f_classif` | 分類 | クラス間で特徴量の平均に差があるか |
| `mutual_info_classif` | 分類 | 特徴量とクラスの依存関係 |
| `f_regression` | 回帰 | 特徴量と連続値ターゲットの線形関係 |
| `mutual_info_regression` | 回帰 | 特徴量と連続値ターゲットの依存関係 |
| `chi2` | 分類 | 非負の特徴量とクラスの関連 |

つまり、`SelectKBest` は「上位 `k` 個を選ぶ仕組み」で、`score_func` は「何を基準に上位を決めるか」です。分類では `f_classif` や `mutual_info_classif`、回帰では `f_regression` や `mutual_info_regression` のように、目的変数の種類に合わせて選びます。

一方、`RandomForestClassifier` は分類モデルそのものです。モデルを学習したあとで、どの特徴量が分類に多く使われたかを `feature_importances_` で確認します。

整理すると、次のように使い分けます。

- `SelectKBest`：特徴量を事前にスコアで選ぶ
- `f_classif`：分類ラベルとの関係を数値化する
- `RandomForestClassifier`：モデルを学習し、その中での重要度を見る

> 注意：特徴量重要度は「因果関係」を示すものではありません。重要度が高い特徴量は、あくまでこのデータとこのモデルで分類に役立った特徴量です。

### 7-3. SelectKBestで上位特徴量を選ぶ

ここでは `type` の赤/白分類を題材にします。第12回で扱った教師あり学習に戻りますが、目的は予測モデルそのものではなく「どの特徴量がラベルと関係しているか」を見ることです。


In [ ]:
selector = SelectKBest(score_func=f_classif, k=5)
selector.fit(X, y_type)

score_df = pd.DataFrame({
    "feature": feature_cols,
    "score": selector.scores_,
    "selected": selector.get_support()
}).sort_values("score", ascending=False)

score_df


In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=score_df, x="score", y="feature", hue="selected", dodge=False)
plt.title("SelectKBestによる特徴量スコア")
plt.xlabel("ANOVA F-value")
plt.ylabel("特徴量")
plt.show()


### 7-4. Random Forestの特徴量重要度


In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)
rf.fit(X, y_type)

importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

importance_df


In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=importance_df, x="importance", y="feature")
plt.title("Random Forestの特徴量重要度")
plt.xlabel("Importance")
plt.ylabel("特徴量")
plt.show()


Random Forestの重要度は、決定木の分割でどれだけ不純度を減らしたかに基づきます。相関の強い特徴量が複数ある場合、重要度が分散することがあります。

---

## 8. 特徴量を減らしてモデルを比較する

最後に、全特徴量を使った場合と、選択した特徴量だけを使った場合で分類性能を比較します。


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_type,
    test_size=0.2,
    random_state=42,
    stratify=y_type
)

selected_features = score_df.query("selected")["feature"].tolist()

models = {
    "all_features": X_train.columns.tolist(),
    "selected_features": selected_features,
}

results = []

for name, cols in models.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ])
    pipe.fit(X_train[cols], y_train)
    pred = pipe.predict(X_test[cols])
    proba = pipe.predict_proba(X_test[cols])[:, 1]

    results.append({
        "setting": name,
        "n_features": len(cols),
        "accuracy": accuracy_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
        "features": ", ".join(cols),
    })

pd.DataFrame(results)


特徴量を減らしても性能がほとんど変わらない場合、少ない特徴量で説明しやすいモデルを作れる可能性があります。一方、性能が大きく下がる場合は、削除した特徴量にも重要な情報が含まれていたと考えられます。

---

## 9. 演習

ここからは、本文で使った `winequality.csv` ではなく、`scikit-learn` に含まれる Breast Cancer Wisconsin データセットを使います。

本文と同じデータで同じ処理を繰り返すのではなく、別のデータに同じ考え方を応用する練習です。

### 9-1. 使用データの準備

Breast Cancer Wisconsin データセットは、細胞核の半径、面積、滑らかさなどの特徴量から、腫瘍が悪性か良性かを扱う練習用データです。


In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()

bc_df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
bc_df["target"] = cancer.target
bc_df["target_name"] = bc_df["target"].map({
    0: cancer.target_names[0],
    1: cancer.target_names[1],
})

print(bc_df.shape)
bc_df.head()


確認すること：

- サンプル数と特徴量数
- `target_name` の内訳
- 特徴量名にどのような単語が多いか

### 9-2. K-meansでラベルなしに分ける

`target` はいったん使わず、特徴量だけで `K=2` のクラスタリングを行ってください。そのあとで、クラスタと実際の `target_name` がどの程度対応しているかを確認します。

ヒント：

- `bc_X` には特徴量だけを入れる
- `StandardScaler()` で標準化する
- `KMeans(n_clusters=2, random_state=42, n_init=10)` を使う
- `pd.crosstab()` でクラスタと実際のラベルを比較する


In [ ]:
# 特徴量 X と目的ラベル y を作る
bc_X = ...
bc_y = ...

# 標準化する
bc_scaler = ...
bc_X_scaled = ...

# K-meansで2つのクラスタに分ける
bc_kmeans = ...
bc_cluster = ...

# 結果を元データに追加する
bc_result = bc_df.copy()
bc_result["cluster"] = ...

# クラスタと実際のラベルを比較する
pd.crosstab(..., ..., normalize="index")


考えること：

- クラスタは悪性・良性ときれいに対応しましたか
- K-meansは `target_name` を見ていないのに、なぜ対応する場合があるのでしょうか
- 対応しないサンプルがあるとしたら、どのような理由が考えられますか

### 9-3. PCAで2次元に可視化する

Breast Cancer データをPCAで2次元に圧縮し、実際のラベルとK-meansクラスタの両方で色分けしてください。

ヒント：

- `PCA(n_components=2, random_state=42)` を使う
- `fit_transform()` で2次元座標を作る
- `explained_variance_ratio_` で寄与率を確認する
- `sns.scatterplot()` の `hue` を変えると、同じ座標を別のラベルで色分けできる


In [ ]:
bc_pca = ...
bc_X_pca = ...

bc_pca_df = pd.DataFrame(bc_X_pca, columns=["PC1", "PC2"])
bc_pca_df["target_name"] = bc_df["target_name"].values
bc_pca_df["cluster"] = bc_cluster

print("寄与率:", ...)
print("累積寄与率:", ...)


In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=..., x="PC1", y="PC2", hue=..., alpha=0.8)
plt.title("Breast Cancer PCA（色：実際のラベル）")
plt.show()

plt.figure(figsize=(7, 5))
sns.scatterplot(data=..., x="PC1", y="PC2", hue=..., palette="Set2", alpha=0.8)
plt.title("Breast Cancer PCA（色：K-meansクラスタ）")
plt.show()


考えること：

- PCAの第1・第2主成分だけで、どの程度ラベルが分かれて見えますか
- K-meansクラスタと実際のラベルの図は似ていますか
- 2次元に圧縮したことで、失われている情報はありそうですか

### 9-4. t-SNEで見え方を比較する

同じデータをt-SNEでも2次元に可視化し、PCAと見え方を比較してください。

ヒント：

- `TSNE(n_components=2, perplexity=30, learning_rate="auto", init="pca", random_state=42)` を使う
- t-SNEは軸の意味ではなく、近いデータ同士のまとまりを見る
- PCAの図と比べて、まとまりが強調されているかを確認する


In [ ]:
bc_tsne = TSNE(
    n_components=...,
    perplexity=...,
    learning_rate=...,
    init=...,
    random_state=...
)
bc_X_tsne = ...

bc_tsne_df = pd.DataFrame(bc_X_tsne, columns=["TSNE1", "TSNE2"])
bc_tsne_df["target_name"] = ...

plt.figure(figsize=(7, 5))
sns.scatterplot(data=..., x="TSNE1", y="TSNE2", hue=..., alpha=0.8)
plt.title("Breast Cancer t-SNE（色：実際のラベル）")
plt.show()


考えること：

- PCAとt-SNEのどちらが、まとまりを見つけやすいですか
- PCAとt-SNEのどちらが、軸の意味を説明しやすいですか
- t-SNEの図だけを見て「悪性・良性を完全に分類できる」と言い切ってよいでしょうか

### 9-5. 特徴量選択を行う

`SelectKBest` を使って、悪性・良性の分類に関係が強そうな特徴量を10個選んでください。

ヒント：

- 分類問題なので `score_func=f_classif` を使う
- `k=10` で上位10特徴量を選ぶ
- `selector.scores_` と `selector.get_support()` を使って結果表を作る
- スコア上位の特徴量名に共通点があるかを見る


In [ ]:
bc_selector = ...
bc_selector.fit(..., ...)

bc_score_df = pd.DataFrame({
    "feature": cancer.feature_names,
    "score": ...,
    "selected": ...
}).sort_values("score", ascending=False)

bc_score_df.head(...)


考えること：

- 上位にはどのような特徴量が多いですか
- `mean radius`、`worst radius`、`mean area` のように、似た意味の特徴量が複数選ばれていませんか
- 特徴量選択では、予測性能だけでなく、説明しやすさも重要です。10個は多すぎますか、少なすぎますか

---

## 🏆 本日の総合課題

Breast Cancer Wisconsin データセットを使って、短い分析レポートを作成してください。本文と同じコードをそのまま写すのではなく、結果を見て自分の言葉で解釈することを重視します。

含める内容：

1. データの概要：サンプル数、特徴量数、ラベルの内訳
2. K-means：`K=2` のクラスタと実際のラベルの対応
3. PCA：2次元散布図と累積寄与率の説明
4. t-SNE：PCAとの見え方の違い
5. 特徴量選択：`SelectKBest` で選ばれた上位10特徴量の特徴
6. 考察：教師なし学習だけで診断ラベルを決めることの危険性

最後に、次の問いに3〜5行で答えてください。

> PCAやt-SNEの図で悪性・良性が分かれて見えた場合、それだけで診断モデルとして十分だと言えるでしょうか。理由も含めて説明してください。

---

## 10. まとめ

今回は、教師なし学習、次元圧縮、特徴量選択を扱いました。

- 教師なし学習は、正解ラベルなしでデータの構造を探す方法
- K-meansは、似ているサンプルをクラスタに分ける代表的手法
- PCAは、多数の特徴量を少数の主成分に要約して可視化する方法
- t-SNEは、近いデータ同士のまとまりを2次元で見やすくする方法
- 特徴量選択は、予測性能と解釈しやすさのバランスを取るために使う
- 教師なし学習の結果は、後から既知ラベルやドメイン知識と照合して解釈する必要がある

第12回までの分類・回帰では、モデルの予測性能を評価しました。第13回では、その前後に行う「データを理解する」「特徴量を整理する」作業を学びました。実際の分析では、PCAで解釈し、t-SNEでまとまりを眺め、特徴量選択でモデルを整理することで、モデルの解釈や改善につなげやすくなります。
